In [ ]:
print("Hello")

// Import Spark classes
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.Dataset

// Create SparkSession
val spark = SparkSession.builder().appName("Ex1").master("local[*]").getOrCreate()

// Import implicits for Dataset encoders
import spark.implicits._



Intitializing Scala interpreter ...

In [2]:
import org.apache.log4j.{Level, Logger}
import org.apache.spark.sql.{DataFrame}
Logger.getRootLogger.setLevel(Level.WARN)

val customerDF: DataFrame = spark.read.option("header","true").csv("/opt/spark/work-dir/ex1/customer_data.csv")
val accountDF = spark.read.option("header","true").csv("/opt/spark/work-dir/ex1/account_data.csv")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

import org.apache.log4j.{Level, Logger}
import org.apache.spark.sql.DataFrame
customerDF: org.apache.spark.sql.DataFrame = [customerId: string, forename: string ... 1 more field]
accountDF: org.apache.spark.sql.DataFrame = [customerId: string, accountId: string ... 1 more field]


In [3]:
case class AccountData          (customerId: String, 
                                                     accountId: String,
                                                     balance: Long  )

case class CustomerData         (customerId: String, forename: String, surname: String)
case class CustomerAccountOutput(customerId: String, forename: String, surname: String,
                                 //Accounts for this customer
                                 accounts: Seq[AccountData],
                                 //Statistics of the accounts
                                 numberAccounts: Int,
                                 totalBalance: Long,
                                 averageBalance: Double
                              )

//Create Datasets of sources
val customerDS: Dataset[CustomerData] = customerDF.as[CustomerData]
val accountDS: Dataset[AccountData] = accountDF.withColumn("balance",'balance.cast("long")).as[AccountData]

customerDS.show
accountDS.show

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

defined class AccountData
defined class CustomerData
defined class CustomerAccountOutput
customerDS: org.apache.spark.sql.Dataset[CustomerData] = [customerId: string, forename: string ... 1 more field]
accountDS: org.apache.spark.sql.Dataset[AccountData] = [customerId: string, accountId: string ... 1 more field]
+----------+-----------+--------+
|customerId|   forename| surname|
+----------+-----------+--------+
|   IND0001|Christopher|   Black|
|   IND0002|  Madeleine|    Kerr|
|   IND0003|      Sarah| Skinner|
|   IND0004|     Rachel| Parsons|
|   IND0005|     Oliver|Johnston|
|   IND0006|       Carl|Metcalfe|
|   IND0007|     Amelia|  Walker|
|   IND0008|      Boris|  Graham|
|   IND0009|     Adrian|Metcalfe|
|   IND0010|      Diana|    Hill|
|   IND0011|       Jake|  Watson|
|   IND0012|    Abigail|   Berry|
|   IND0013|        Zoe|     Lee|
|   IND0014|  Gabrielle|    Ince|
|   IND0015|     Lauren|    Reid|
|   IND0016|    Stephen|   Lyman|
|   IND0017|     Oliver|Hamilton|
|   IN

In [4]:
// Group accounts by customerId and aggregate statistics

// val accountsByCustomer = accountDS.groupByKey(_.customerId).mapGroups { (customerId, accounts) =>
//   val accountSeq = accounts.toSeq
//   val numAccounts = accountSeq.length
//   val total = accountSeq.map(_.balance).sum
//   val avg = if (numAccounts > 0) total.toDouble / numAccounts else 0.0
//   (customerId, accountSeq, numAccounts, total, avg)
// }
// accountsByCustomer.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
  // Join with customer data (left join to include all customers)

  // val customerAccountOutputDS: Dataset[CustomerAccountOutput] = customerDS.joinWith(
  //     accountsByCustomer, customerDS("customerId") === accountsByCustomer("_1"), "left").map {
  //     case (customer, accountInfo) =>
  //       if (accountInfo != null) {
  //         CustomerAccountOutput(
  //           customer.customerId,
  //           customer.forename,
  //           customer.surname,
  //           accountInfo._2,
  //           accountInfo._3,
  //           accountInfo._4,
  //           accountInfo._5
  //         )
  //       } else {
  //         // Customer with no accounts
  //         CustomerAccountOutput(
  //           customer.customerId,
  //           customer.forename,
  //           customer.surname,
  //           Seq.empty,
  //           0,
  //           0L,
  //           0.0
  //         )
  //       }
  //   }

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Change to use `Seq(...)`

In [6]:
// create join key for extendability
val joinCustAccColumns = Seq("customerId")
val joinCustAccOnKey = joinCustAccColumns.map(col => customerDS(col) === accountDS(col)).reduce(_ && _)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

joinCustAccColumns: Seq[String] = List(customerId)
joinCustAccOnKey: org.apache.spark.sql.Column = (customerId = customerId)


In [7]:
val customerAccountOutputDS_test = customerDS.joinWith(accountDS, joinCustAccOnKey, "left")
customerAccountOutputDS_test.show(2)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

customerAccountOutputDS_test: org.apache.spark.sql.Dataset[(CustomerData, AccountData)] = [_1: struct<customerId: string, forename: string ... 1 more field>, _2: struct<customerId: string, accountId: string ... 1 more field>]
+--------------------+--------------------+
|                  _1|                  _2|
+--------------------+--------------------+
|{IND0001, Christo...|                null|
|{IND0002, Madelei...|{IND0002, ACC0262...|
+--------------------+--------------------+
only showing top 2 rows



In [8]:
customerAccountOutputDS_test.head(2)(1)._2.accountId

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

res56: String = ACC0262


In [9]:
val customerAccountOutputDS: Dataset[CustomerAccountOutput] = customerDS.joinWith(
    accountDS, joinCustAccOnKey, "left").groupByKey { 
    case (customer, _) => customer }.mapGroups { (customer, rows) =>
      val accounts = rows.flatMap { case (_, account) => Option(account) }.toSeq
      val numAccounts = accounts.length
      val total = accounts.map(_.balance).sum
      val avg = if (numAccounts > 0) total.toDouble / numAccounts else 0.0
      
      CustomerAccountOutput(
        customer.customerId,
        customer.forename,
        customer.surname,
        accounts,
        numAccounts,
        total,
        avg
      )
    }




FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

customerAccountOutputDS: org.apache.spark.sql.Dataset[CustomerAccountOutput] = [customerId: string, forename: string ... 5 more fields]


In [10]:
  customerAccountOutputDS.show(2, truncate = false)
  customerAccountOutputDS.write.mode("overwrite").parquet("src/main/resources/customerAccountOutputDS.parquet")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 11.0 failed 1 times, most recent failure: Lost task 0.0 in stage 11.0 (TID 10) (4075a435c667 executor driver): java.lang.ClassCastException: class CustomerAccountOutput cannot be cast to class CustomerAccountOutput (CustomerAccountOutput is in unnamed module of loader org.apache.spark.repl.ExecutorClassLoader @5ef7b26b; CustomerAccountOutput is in unnamed module of loader scala.tools.nsc.interpreter.IMain$TranslatingClassLoader @3e7f02a2)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage4.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenExec$$anon$1.hasNext(WholeStageCodegenExec.scala:760)
	at org.apache.spark.sql.execution.SparkPlan.$anonfun$getByteArrayRdd$1(SparkPlan.scala:364)
	at org.apache.spark.rdd.RDD.$anon

In [11]:
spark.stop()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
// Define the Car case class
case class Car(make: String, model: String, year: Int, price: Double)

// Create the dataset
val cars: Dataset[Car] = spark.createDataset(Seq(
  Car("Toyota", "Camry", 2020, 25000),
  Car("Toyota", "Corolla", 2020, 22000),
  Car("Honda", "Civic", 2020, 23000),
  Car("Honda", "Accord", 2020, 28000)
))

case class CarSummary(make: String, models: String, avgPrice: Double)

// Group by make and use mapGroups to create custom aggregation
val carsByMake = cars.groupByKey(_.make).mapGroups { (make, cars) =>
    val carList = cars.toList
    val avgPrice = carList.map(_.price).sum / carList.length
    val models = carList.map(_.model).mkString(", ")
    CarSummary(make, models, avgPrice)
  }

// Show the results
carsByMake.show(false)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.SparkContext.getOrCreate(SparkContext.scala)
org.apache.livy.rsc.driver.SparkEntries.sc(SparkEntries.java:52)
org.apache.livy.rsc.driver.SparkEntries.sparkSession(SparkEntries.java:66)
org.apache.livy.repl.AbstractSparkInterpreter.postStart(AbstractSparkInterpreter.scala:71)
org.apache.livy.repl.SparkInterpreter.$anonfun$start$1(SparkInterpreter.scala:85)
scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.java:23)
org.apache.livy.repl.AbstractSparkInterpreter.restoreContextClassLoader(AbstractSparkInterpreter.scala:342)
org.apache.livy.repl.SparkInterpreter.start(SparkInterpreter.scala:60)
org.apache.livy.repl.Session.$anonfun$start$1(Session.scala:128)
scala.concurrent.Future$.$anonfun$apply$1(Future.scala:659)
scala.util.Success.$anonfun$map$1(Try.scala:255)
scala.util.Success.map(Try.scala:213)
scala.concurre

In [13]:
spark.stop()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…